In [1]:
import regex as re
from tqdm import tqdm

class RegexTokenizer:
    def __init__(self):
        self.vocab = {}
        self.merges = {}
        self.reverse_vocab = {}
        self.pattern = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        self.compiled_pattern = re.compile(self.pattern)
        
        
    def get_stats(self, ids, counts = None):
        counts = {} if counts is None else counts
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        
        return counts

    def merge_pair(self, ids, pair, new_idx):
        new_ids = []
        i = 0
        
        while i<len(ids):
            
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1]==pair[1]:
                new_ids.append(new_idx)
                i+=2
            
            else:
                new_ids.append(ids[i])
                i+=1
        
        return new_ids

    def train(self, text, vocab_size, verbose=False):
        
        assert vocab_size >= 256, "vocab_size must be at least 256 to accommodate the initial byte tokens"
        number_of_merges = vocab_size - 256
        
        pattern = self.compiled_pattern
        
        token_chunks = pattern.findall(text) # use the regex pattern to split the text into tokens, which will be the initial tokens for the BPE algorithm
        # print(token_chunks)
        
        ids = [list(chunk.encode("utf-8")) for chunk in token_chunks] # convert each token chunk to bytes and then to a list of integers for processing
        
        # print(ids)
        
        # merges = {} # (int, int) -> int
        
        
        # initialize the reverse vocab with the original 256 tokens, where each token is just a single byte corresponding to its index e.g. reverse_vocab[104] = b'h' so in the dictionary {104 : b'h'} and its technically a reverse vocab since it maps from token index to token bytes, but we can call it reverse_vocab for simplicity
        self.reverse_vocab = { idx : bytes([idx]) for idx in range(256)} # idx -> bytes 
        
        for i in tqdm(range(number_of_merges), desc="Training BPE"):
            # count the number of times every consecutive pair appears
            stats = {}
            
            for chunk_ids in ids:
                self.get_stats(chunk_ids, counts=stats) # get the stats for this chunk and add it to the overall stats for this iteration
                
            if not stats:
                if verbose:
                    print("No more pairs to merge, stopping early.")
                break # no more pairs to merge
            
            # find the pair with the highest count
            # O(P) max() where P = number of unique pairs.
            pair = max(stats, key = stats.get)
                
            idx = 256 + i
            
            # replace all occurrences of pair in ids with idx    
            ids = [self.merge_pair(chunk_ids, pair, idx) for chunk_ids in ids] # merge the pair in each chunk
            
            self.merges[pair] = idx
            self.reverse_vocab[idx] = self.reverse_vocab[pair[0]] + self.reverse_vocab[pair[1]] # new token is concatenation of merged tokens
            
            if verbose and i < 20: # print the first 20 merges for debugging and insight into the training process, after that it gets too verbose and not very informative to print every merge
                print(f"merge {i+1}/{number_of_merges}: {pair} -> {idx} ({self.reverse_vocab[idx]}) had {stats[pair]} occurrences")

        
        self.vocab = {v : k for k,v in self.reverse_vocab.items()} # not needed for encoding since we can just use the merges to merge pairs in the input tokens, but could be useful for debugging or other purposes and we dont use it in this implementation since we can just use the merges to merge pairs in the input tokens, but could be useful for debugging or other purposes 
        
    def encode_chunks(self, chunk):
        
        chunk_ids = list(map(int, chunk)) # convert the chunk to a list of integers for processing
        
        while len(chunk_ids) >= 2:
                stats = self.get_stats(chunk_ids)
                pair = min(stats, key = lambda p : self.merges.get(p, float('inf'))) # find the pair with the smallest new index in merges that still exists in ids because we need to apply merges in the same order they were created for correct encoding
            
                if pair not in self.merges:
                    break # no more pairs to merge, we’re done
                
                # print(pair)
                
                chunk_ids = self.merge_pair(chunk_ids, pair, self.merges[pair]) # merge the pair in the ids list
        
        return chunk_ids

    def encode(self, text):
        
        token_chunks = self.compiled_pattern.findall(text) # use the regex pattern to split the text into tokens, which will be the initial tokens for the BPE algorithm
        
        ids = [] # all chunks of text are encoded separately, then results are joined
        
        for token_chunk in token_chunks:
            chunk_bytes = token_chunk.encode("utf-8") # convert the chunk to bytes
            chunk_ids = self.encode_chunks(chunk_bytes) # encode the chunk using the same merges that were learned during training, which will give us a list of token ids for this chunk
            ids.extend(chunk_ids) # add the token ids for this chunk to the overall list of token ids for the whole input text
        
        return ids
    
    def decode(self, ids):
        text = bytes().join(self.reverse_vocab[idx] for idx in ids) # get the byte sequence for each token index and concatenate them, using an empty byte string for any unknown indices (shouldn’t happen if encoding was done with the same vocab)
        return text.decode("utf-8", errors="replace") # decode the bytes back to a string, replacing any invalid sequences with the Unicode replacement character
    

In [2]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

C:\Users\biswa\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train = dataset["train"]

In [4]:
text = "\n".join(
    line for line in train["text"]
    if line.strip()
)

In [5]:
training_text_10mb = text[:10_000_000]  # first 10 MB

with open("wikitext_10MB.txt", "w", encoding="utf-8") as f:
    f.write(training_text_10mb)

In [6]:
training_text_50mb = text[:50_000_000]  # first 50 MB

with open("wikitext_50MB.txt", "w", encoding="utf-8") as f:
    f.write(training_text_50mb)

In [7]:
regexTokenizer10mb = RegexTokenizer()

# with open("TinyStoriesV2-GPT4-train.txt", "r", encoding="utf-8") as f:
#     text = f.read() # read the first 50 million characters for training, which should be enough to learn a good BPE vocab without taking too long to train    


In [ ]:
regexTokenizer10mb.train(training_text_50mb, vocab_size=4096, verbose=False)

Training BPE:   0%|          | 18/3840 [10:29<33:11:46, 31.27s/it]

In [34]:
model_file = "bpe_tokenizer_10MB" + ".model"
with open(model_file, "w") as f:
    f.write("bpe_tokenizer_v1_10MB\n")
    f.write(f"{regexTokenizer10mb.pattern}\n")

# implement when we have special tokens   
    # f.write(f"{len(regexTokenizer10mb.special_tokens)}\n")
    # for special, idx in regexTokenizer10mb.special_tokens.items():
    #     f.write(f"{special} {idx}\n")
    
    for idx1, idx2 in regexTokenizer10mb.merges:
        f.write(f"{idx1} {idx2}\n")


In [35]:
vocab_file = "bpe_tokenizer_10MB" + ".vocab"
inverted_merges = {v:k for k,v in regexTokenizer10mb.merges.items()} # idx -> (idx1, idx2)
with open(vocab_file, "w", encoding = "utf-8") as f:
    for idx, token in regexTokenizer10mb.reverse_vocab.items():
        s = regexTokenizer10mb.decode([idx]) # decode the token bytes back to a string for saving in the vocab file, which will be useful for debugging and understanding what each token index corresponds to in terms of text
        if(idx in inverted_merges):
            idx1, idx2 = inverted_merges[idx]
            s1 = regexTokenizer10mb.decode([idx1])
            s2 = regexTokenizer10mb.decode([idx2])
            f.write(f"[{s1}][{s2}] -> [{s}] {idx}\n")
        else:
            f.write(f"[{s}] {idx}\n")

In [2]:
text = "hello hello hello"

encoded = regexTokenizer10mb.encode(text)  # fix encode and decode for regex logic
decoded = regexTokenizer10mb.decode(encoded)

print(encoded)
print(encoded == regexTokenizer10mb.encode(text)) # should be True
print(len(encoded))
print(decoded)

NameError: name 'regexTokenizer10mb' is not defined

In [37]:
text = "hello world!!!? (안녕하세요!) lol123 😉"

encoded = regexTokenizer10mb.encode(text)
decoded = regexTokenizer10mb.decode(encoded)

print(encoded)
print(encoded == regexTokenizer10mb.encode(text)) # should be True
print(len(encoded))
print(decoded)

[257, 860, 111, 1310, 33, 33, 33, 63, 374, 236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 33, 41, 311, 332, 881, 51, 32, 240, 159, 152, 137]
True
35
hello world!!!? (안녕하세요!) lol123 😉


In [41]:
# better performance than using the taylor swift dataset which gave me 36 words and the TinyStories dataset which gave me 37 now after modifying the tokenizer logic to be more efficient we will be able to train on larger datasets and get a much richer vocab that can tokenize text more efficiently with fewer tokens per input text which will be very beneficial for training a large language model since it will allow us to fit more text into the context window and also reduce the computational cost of encoding and decoding text during training and inference, which is crucial for training large language models effectively.

In [ ]:
#GPT-4 tokenizer 

import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids)
print(len(ids))
print(enc.decode(ids))
print(enc.n_vocab)

[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
14
hello world!!!? (안녕하세요!) lol123 😉
100277


In [42]:
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

In [33]:
# NOTE - Now we will be training on a more diverse dataset, which should give us a better tokenizer that can handle a wider range of inputs more efficiently, since the initial test on the small text was just to verify that the training and encoding/decoding process works correctly, but it’s not representative of the performance we can expect on real-world data with a much larger and more diverse training corpus, while the text we will import is very large we take only the first 50 million characters for training, which should be enough to learn a good BPE vocab without taking too long to train

In [17]:
# NOTE:- we will now be modifying the code a bit to make it more efficient for large training corpora, since the original implementation is not very efficient and would take a long time to train on a large dataset, we will be using a more efficient way to count pairs and merge them, which should significantly reduce the training time while still learning a good BPE vocab. And we also implement special tokens in the training and encoding/decoding process, which will be useful for handling things like padding, unknown tokens, start/end of sequence, etc. in a more robust way when we use this tokenizer for training a language model later on, since special tokens are an important part of a practical tokenizer for NLP tasks.

import regex as re
from tqdm import tqdm
from collections import defaultdict
import heapq

class OptimizedRegexTokenizer:
    def __init__(self, pattern = None):
        self.vocab = {}
        self.merges = {}
        self.reverse_vocab = {}
        self.pattern = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""" if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.special_tokens = {} # special token string -> index, e.g. {"<PAD>": 4096, "<UNK>": 4097, "<SOS>": 4098, "<EOS>": 4099} for a vocab size of 4100 with 4 special tokens
        self.inverse_special_tokens = {} # index -> special token string, e.g. {4096: "<PAD>", 4097: "<UNK>", 4098: "<SOS>", 4099: "<EOS>"}
        
    def build_initial_stats(self, ids):
        """
        Build pair_counts, pair_index, and the initial heap in one O(N) pass.
 
        pair_counts : Dict[(int,int), int]
            How many times each adjacent pair appears across ALL chunks.
 
        pair_index  : Dict[(int,int), set of chunk_idx]
            Which chunks contain each pair at least once.
            Stores chunk membership only — no exact positions.
 
        heap : List of (-count, pair) tuples — min-heap simulating a max-heap.
            heapq.heapify() runs in O(P). Entries go stale as training proceeds;
            staleness is resolved lazily at pop time in train().
 
        Called ONCE. O(N + P) where N = total tokens, P = unique pairs.
        """
        pair_counts = defaultdict(int)
        pair_index  = defaultdict(set)   # pair -> {chunk_idx, ...}
        
        for chunk_idx, chunk in enumerate(ids):
            for pos in range(len(chunk) - 1):
                pair = (chunk[pos], chunk[pos + 1])
                pair_counts[pair] += 1
                pair_index[pair].add(chunk_idx)
                
        heap = [(-count, pair) for pair, count in pair_counts.items()]
        heapq.heapify(heap)  # O(P)
        
        return pair_counts, pair_index, heap
    
    def update_stats(self, ids, pair, new_idx, pair_counts, pair_index, heap):
        """
        After merging `pair` -> `new_idx`, update pair_counts, pair_index,
        and heap by scanning only the chunks that contained the winning pair.
        
        Algorithm per affected chunk:
          1. Walk the chunk with a while loop.
          2. At each merge site (chunk[i]==a, chunk[i+1]==b):
               - Record delta -1 for old left/right neighbour pairs.
               - Record delta +1 for new left/right neighbour pairs.
               - Splice [a, b] -> [c] in-place.
               - Do NOT advance i — c may form a new pair with chunk[i+1].
          3. After the scan, apply all deltas to pair_counts.
          4. For pairs with net positive delta: add chunk_idx to pair_index,
             push fresh heap entry.
          5. For pairs with net negative delta: scan chunk to confirm whether
             any occurrences remain, then update pair_index membership.
 
        Edge case (a == b, e.g. merging (101,101)):
          The token at i+2 may equal a, forming another occurrence of the
          winning pair. The old_right != pair guard prevents double-decrementing
          it here — the while loop will handle that occurrence when it reaches i.
 
        Returns: Nothing. Mutates ids, pair_counts, pair_index, heap in-place.
        """
        a, b = pair
        c = new_idx
 
        affected_chunks = list(pair_index.get(pair, set()))
        
        # dbug_pair = (111, 110)
        # for chunk_idx in pair_index[dbug_pair]:
        #     chunk = ids[chunk_idx]
        #     count = sum(1 for j in range(len(chunk)-1) if chunk[j] == dbug_pair[0] and chunk[j+1] == dbug_pair[1])
        #     print(f"chunk {chunk_idx}: {count} occurrences")
 
        for chunk_idx in affected_chunks:
            chunk = ids[chunk_idx]
            delta_counts = defaultdict(int)   # net change per neighbour pair
 
            i = 0
            while i < len(chunk) - 1:
                if chunk[i] == a and chunk[i + 1] == b:
 
                    # Left neighbour
                    if i > 0:
                        x = chunk[i - 1]
                        delta_counts[(x, a)] -= 1
                        delta_counts[(x, c)] += 1
 
                    # Right neighbour
                    if i + 2 < len(chunk):
                        y = chunk[i + 2]
                        old_right = (b, y)
                        new_right = (c, y)
                        if old_right != pair:   # edge-case guard
                            delta_counts[old_right] -= 1
                        delta_counts[new_right] += 1
 
                    # Merge in-place; do NOT advance i
                    chunk[i:i + 2] = [c]
 
                else:
                    i += 1
 
            # ---- Apply deltas atomically ------------------------------------
            for p, delta in delta_counts.items():
                if delta == 0:
                    continue
 
                old_count = pair_counts.get(p, 0)
                new_count = old_count + delta
 
                if new_count <= 0:
                    pair_counts.pop(p, None)
                    pair_index.pop(p, None)
                else:
                    pair_counts[p] = new_count
 
                    if delta > 0:
                        # Pair gained occurrences — ensure chunk membership
                        # and push fresh heap entry.
                        pair_index[p].add(chunk_idx)
                        heapq.heappush(heap, (-new_count, p))
                    else:
                        # Pair lost occurrences. Confirm whether it still
                        # exists in this chunk with a targeted scan.
                        # (Cheaper than a full corpus rescan: only this chunk.)
                        still_here = any(
                            chunk[j] == p[0] and chunk[j + 1] == p[1]
                            for j in range(len(chunk) - 1)
                        )
                        if not still_here:
                            pair_index[p].discard(chunk_idx)
                        if not pair_index.get(p):
                            pair_index.pop(p, None)
 
        # Remove the winning pair — it no longer exists in the corpus.
        pair_counts.pop(pair, None)
        pair_index.pop(pair, None)

    def merge_pair(self, ids, pair, new_idx):
        new_ids = []
        i = 0
        
        while i<len(ids):
            
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1]==pair[1]:
                new_ids.append(new_idx)
                i+=2
            
            else:
                new_ids.append(ids[i])
                i+=1
        
        return new_ids
    
    def get_stats(self, ids, counts = None):
        counts = {} if counts is None else counts
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        
        return counts

    def get_stats_all(self, ids):
        """Ground-truth pair counts by full rescan. Use to validate pair_counts."""
        counts = {}
        for chunk in ids:
            self.get_stats(chunk, counts)
        return counts

    def train(self, text, vocab_size, verbose=False, validate=False):
        """
        Train BPE using incremental pair count updates.
 
        validate=True runs a full rescan after each merge and asserts that
        pair_counts matches ground truth. Useful for debugging; disable for
        production since it re-introduces the O(N) rescan per iteration.
        """
        
        assert vocab_size >= 256, "vocab_size must be >= 256"
        number_of_merges = vocab_size - 256
        
        token_chunks = self.compiled_pattern.findall(text) # use the regex pattern to split the text into tokens, which will be the initial tokens for the BPE algorithm
        # print(token_chunks)
        
        ids = [list(chunk.encode("utf-8")) for chunk in token_chunks] # convert each token chunk to bytes and then to a list of integers for processing
        
        # print(ids)
        
        # merges = {} # (int, int) -> int
        
        
        # initialize the reverse vocab with the original 256 tokens, where each token is just a single byte corresponding to its index e.g. reverse_vocab[104] = b'h' so in the dictionary {104 : b'h'} and its technically a reverse vocab since it maps from token index to token bytes, but we can call it reverse_vocab for simplicity
        self.reverse_vocab = {idx : bytes([idx]) for idx in range(256)} # idx -> bytes 
        
        # ONE-TIME O(N) BUILD of pair_counts and pair_index
        pair_counts, pair_index, heap = self.build_initial_stats(ids)
        
        
        for i in tqdm(range(number_of_merges), desc="Training BPE"):
            
            if not pair_counts:
                if verbose:
                    print("No more pairs to merge, stopping early.")
                break
            
            # O(log P) where P = number of unique pairs, which is much more efficient than O(P) for max() in the original implementation
            while True:
                if not heap:
                    break  # no pairs left in the heap
                # Validity check: does the stored count match ground truth?
                # -neg_count converts back from the min-heap negation trick.
                neg_count, pair = heapq.heappop(heap)
                if pair_counts.get(pair, 0) == -neg_count:
                    break  # found the most frequent valid pair
                # else this pair is stale (count has changed), skip it and continue popping
                
            idx = 256 + i
            
            # print("\n" + "="*80)
            # print(f"MERGE {i}")
            # print(f"Winning pair: {pair}")
            # print(f"Count according to pair_counts: {pair_counts[pair]}")
            # print(f"Occurrences stored: {sorted(pair_index[pair])}")
            # print(f"Actual number of occurrences in pair_index:{len(pair_index[pair])}" )
            # print(f"Chunk states before merge:")
            # for chunk in ids:
            #     print(f"  {chunk}")
                
            # Incremental update: O(k_i) where k_i = occurrences of this pair
            self.update_stats(ids, pair, idx, pair_counts, pair_index, heap)
            
            self.merges[pair] = idx
            self.reverse_vocab[idx] = (self.reverse_vocab[pair[0]] + self.reverse_vocab[pair[1]]) # new token is concatenation of merged tokens
            
            if verbose and i < 20:
                print(f"merge {i+1}/{number_of_merges}: {pair} -> {idx} "
                      f"({self.reverse_vocab[idx]})")
 
            # Optional: validate pair_counts against ground truth
            if validate:
                ground_truth = self.get_stats_all(ids)
                # Normalise: remove zero-count entries from pair_counts
                live_counts = {k: v for k, v in pair_counts.items() if v > 0}
                extra = set(live_counts) - set(ground_truth)
                missing = set(ground_truth) - set(live_counts)

                if extra or missing:
                    print("\n" + "!"*80)
                    print("MISMATCH DETECTED")
                    print("EXTRA:", extra)
                    print("MISSING:", missing)

                    for pair in sorted(set(live_counts) | set(ground_truth)):
                        if live_counts.get(pair,0) != ground_truth.get(pair,0):
                            print(
                                pair,
                                "incremental=",
                                live_counts.get(pair,0),
                                "truth=",
                                ground_truth.get(pair,0)
                            )
                            
                assert live_counts == ground_truth, (
                    f"Iteration {i}: pair_counts drifted from ground truth!\n"
                    f"  Extra in pair_counts:  {set(live_counts) - set(ground_truth)}\n"
                    f"  Missing from pair_counts: {set(ground_truth) - set(live_counts)}"
                )

        
        self.vocab = {v : k for k,v in self.reverse_vocab.items()} # not needed for encoding since we can just use the merges to merge pairs in the input tokens, but could be useful for debugging or other purposes and we dont use it in this implementation since we can just use the merges to merge pairs in the input tokens, but could be useful for debugging or other purposes 
        
    def encode_chunks(self, chunk):
        
        chunk_ids = list(map(int, chunk)) # convert the chunk to a list of integers for processing
        
        while len(chunk_ids) >= 2:
                stats = self.get_stats(chunk_ids)
                pair = min(stats, key = lambda p : self.merges.get(p, float('inf'))) # find the pair with the smallest new index in merges that still exists in ids because we need to apply merges in the same order they were created for correct encoding
            
                if pair not in self.merges:
                    break # no more pairs to merge, we’re done
                
                # print(pair)
                
                chunk_ids = self.merge_pair(chunk_ids, pair, self.merges[pair]) # merge the pair in the ids list
        
        return chunk_ids

    def encode(self, text):
        
        token_chunks = self.compiled_pattern.findall(text) # use the regex pattern to split the text into tokens, which will be the initial tokens for the BPE algorithm
        
        ids = [] # all chunks of text are encoded separately, then results are joined
        
        for token_chunk in token_chunks:
            chunk_bytes = token_chunk.encode("utf-8") # convert the chunk to bytes
            chunk_ids = self.encode_chunks(chunk_bytes) # encode the chunk using the same merges that were learned during training, which will give us a list of token ids for this chunk
            ids.extend(chunk_ids) # add the token ids for this chunk to the overall list of token ids for the whole input text
        
        return ids
    
    # def encode(self, text):
    #     pass
    
    def decode(self, ids):
        text = bytes().join(self.reverse_vocab[idx] for idx in ids) # get the byte sequence for each token index and concatenate them, using an empty byte string for any unknown indices (shouldn’t happen if encoding was done with the same vocab)
        return text.decode("utf-8", errors="replace") # decode the bytes back to a string, replacing any invalid sequences with the Unicode replacement character
    

In [31]:
train_text = "\n".join(
    line for line in train["text"]
    if line.strip()
)

In [32]:
training_text_100mb = train_text[:100_000_000]  # first 100 MB

In [33]:
tok = OptimizedRegexTokenizer()
tok.train(training_text_100mb, vocab_size=8192, validate=False, verbose=False)

Training BPE: 100%|██████████| 7936/7936 [11:10<00:00, 11.83it/s] 


In [34]:
text = "hello hello hello"

encoded = tok.encode(text)  # fix encode and decode for regex logic
decoded = tok.decode(encoded)

print(encoded)
print([tok.decode([idx]) for idx in encoded])
print(encoded == tok.encode(text)) # should be True
print(len(encoded))
print(decoded)

[3104, 108, 111, 1467, 108, 111, 1467, 108, 111]
['hel', 'l', 'o', ' hel', 'l', 'o', ' hel', 'l', 'o']
True
9
hello hello hello


In [36]:
text = "hello world!!!? (안녕하세요!) lol123 😉"

encoded = tok.encode(text)
decoded = tok.decode(encoded)

print(encoded)
print([tok.decode([idx]) for idx in encoded])
print(encoded == tok.encode(text)) # should be True
print(len(encoded))
print(decoded)

[3104, 108, 111, 1408, 33, 33, 33, 63, 353, 236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 33, 41, 32, 108, 111, 108, 888, 51, 32, 240, 159, 152, 137]
['hel', 'l', 'o', ' world', '!', '!', '!', '?', ' (', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '!', ')', ' ', 'l', 'o', 'l', '12', '3', ' ', '�', '�', '�', '�']
True
37
hello world!!!? (안녕하세요!) lol123 😉


In [40]:
text = "strawberry"

encoded = tok.encode(text)
decoded = tok.decode(encoded)

print(encoded)
print([tok.decode([idx]) for idx in encoded])
print(encoded == tok.encode(text)) 
print(len(encoded))
print(decoded)

[115, 2464, 119, 98, 101, 2295]
['s', 'tra', 'w', 'b', 'e', 'rry']
True
6
strawberry


In [37]:
#GPT-4 tokenizer 

import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids)
print(len(ids))
print(enc.decode(ids))
print(enc.n_vocab)

[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
14
hello world!!!? (안녕하세요!) lol123 😉
100277


In [42]:
model_file = "bpe_tokenizer_100MB" + ".model"
with open(model_file, "w") as f:
    f.write("bpe_tokenizer_v1_100MB\n")
    f.write(f"{tok.pattern}\n")

# implement when we have special tokens   
    # f.write(f"{len(regexTokenizer100mb.special_tokens)}\n")
    # for special, idx in regexTokenizer100mb.special_tokens.items():
    #     f.write(f"{special} {idx}\n")
    
    for idx1, idx2 in tok.merges:
        f.write(f"{idx1} {idx2}\n")

In [43]:
vocab_file = "bpe_tokenizer_100MB" + ".vocab"
inverted_merges = {v:k for k,v in tok.merges.items()} # idx -> (idx1, idx2)
with open(vocab_file, "w", encoding = "utf-8") as f:
    for idx, token in tok.reverse_vocab.items():
        s = tok.decode([idx]) # decode the token bytes back to a string for saving in the vocab file, which will be useful for debugging and understanding what each token index corresponds to in terms of text
        if(idx in inverted_merges):
            idx1, idx2 = inverted_merges[idx]
            s1 = tok.decode([idx1])
            s2 = tok.decode([idx2])
            f.write(f"[{s1}][{s2}] -> [{s}] {idx}\n")
        else:
            f.write(f"[{s}] {idx}\n")

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset("Skylion007/openwebtext")

# parts = []
# current_size = 0

# for text in dataset["train"]["text"]:
#     text = text.strip()

#     parts.append(text)
#     current_size += len(text) + 1

#     if current_size >= 50_000_000:
#         break

# text50MB = "\n".join(parts)

# with open("openwebtext_50MB.txt", "w", encoding="utf-8") as f:
#     f.write(text50MB)

C:\Users\biswa\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
